In [ ]:
import os
import zipfile
import io

def clean_zip_csv(zip_path: str, output_zip_path: str):
    """
    Removes an extraneous datetime stamp from the first data row of a CSV inside a ZIP archive, as well as 
    extraneous \n characters from the end. Keeps the ZIP archive intact except for the fixed CSV file.

    Explanation: Due to an error in the ae31.py _save_data function prior to 2025-02-26, ae31*.csv files 
    inside the ae31*.zip archives have an extraneous date time stamp following the header line and an 
    extraneous \n character at the end of the file.
    
    :param zip_path: Path to the input ZIP file.
    :param output_zip_path: Path to the output cleaned ZIP file.
    """
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            file_list = zf.namelist()
            csv_files = [f for f in file_list if f.endswith('.csv')]
            
            if not csv_files:
                raise ValueError("No CSV file found in the ZIP archive.")
            
            csv_filename = csv_files[0]
            
            with zf.open(csv_filename) as f:
                lines = f.readlines()
            
            if len(lines) < 2:
                raise ValueError("CSV file does not contain enough lines to check for the error.")
            
            # Decode first two lines
            header = lines[0].decode('utf-8').strip()
            first_data_line = lines[1].decode('utf-8').strip().split(',')
            
            # If first data line has an extra timestamp, remove it
            if len(first_data_line) > 1 and first_data_line[0].startswith("202") and first_data_line[1].startswith("202"):
                fixed_first_data_line = ','.join(first_data_line[1:])  # Remove the first element
            else:
                fixed_first_data_line = lines[1].decode('utf-8').strip()
            
            # Reassemble the CSV content
            fixed_lines = [header, fixed_first_data_line] + [line.decode('utf-8').strip() for line in lines[2:]]
            fixed_lines = '\n'.join(fixed_lines)  # Ensure single newlines between rows
            
            # Write to a new ZIP file
            os.makedirs(os.path.dirname(output_zip_path), exist_ok=True)
            with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zf_out:
                for file in file_list:
                    if file != csv_filename:
                        zf_out.writestr(file, zf.read(file))  # Copy other files as is
                    else:
                        zf_out.writestr(csv_filename, ''.join(fixed_lines))  # Write fixed CSV

    except Exception as err:
        print(zip_path, err)
        pass

In [13]:
source = "nrbdaq/tests/data/ae31"

try:
    for root, _, files in os.walk(source):
        for file in files:
            clean_zip_csv(zip_path=os.path.join(root, file),
                        output_zip_path=os.path.join(root, "fixed", file))
except Exception as err:
    print(err)

nrbdaq/tests/data/ae31/AE31_20240825.csv File is not a zip file
nrbdaq/tests/data/ae31/AE31_20240805.csv File is not a zip file


In [22]:
source = "/product_data/data/pay/Kenya/NRB/incoming/ae31"

try:
    for root, _, files in os.walk(source):
        for file in files:
            clean_zip_csv(zip_path=os.path.join(root, file),
                        output_zip_path=os.path.join(root, file))
except Exception as err:
    print(err)

/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240803.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240804.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240805.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240806.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240807.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240808.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240809.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240809_.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240810.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240811.csv File is not a zip file
/product_data/data/pay/Kenya/NRB/incoming/ae31/2024/AE31_20240812.csv